# Alignment v3 results

This notebook is observational: Slurm jobs write canonical artifacts and execute only the cell associated with their stage. COCO validation results appear only after the final recipe is locked.

In [1]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd()
RESULTS = ROOT / 'results/alignment_v3'
STAGE = os.environ.get('ALIGNMENT_V3_NOTEBOOK_STAGE', 'all')
ARRAY_INDEX = os.environ.get('ALIGNMENT_V3_NOTEBOOK_INDEX')
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None
def read_csv(path):
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()
def pending(message):
    display(Markdown(f'> **Pending:** {message}'))
display(Markdown(f'Notebook stage: `{STAGE}`; array index: `{ARRAY_INDEX}`'))

Notebook stage: `prefetch`; array index: `None`

In [2]:
validation = read_json(RESULTS / 'manifests/validation.json')
split = read_json(RESULTS / 'manifests/split.json')
prefetch = read_json(RESULTS / 'prefetch.json')
display(Markdown('## Validation, leakage controls and model prefetch'))
display(pd.DataFrame([validation]) if validation else Markdown('Validation pending.'))
display(pd.DataFrame([split]) if split else Markdown('Split pending.'))
if prefetch: display(pd.DataFrame([prefetch]))

## Validation, leakage controls and model prefetch

,status,pipeline,split,v2_lock_digest,job_counts,optional_missing,runtime_versions,code_fingerprint
0,READY,configs/alignment_v3/pipeline.yaml,"{'status': 'COMPLETE', 'seed': 3103, 'source_c...",3a1a32d4017034a3fef7f60d1c5b1e93a1ec162179d5bf...,"{'reference': 3, 'pair': 6, 'sensitivity': 8, ...",[data/flickr30k/test.csv],"{'python': '3.9.25', 'platform': 'Linux-5.14.0...",c786fe100bb635a25f087a72f4af4b64285e8de9514f0d...


,status,seed,source_csv,source_sha256,train_csv,dev_csv,train_images,dev_images,train_captions,dev_captions,train_split_hash,dev_split_hash,final_images,final_split_hash,overlap_counts
0,COMPLETE,3103,/mnt/fast/nobackup/scratch4weeks/pp01184/align...,40f96c08486517b3113478a1ea63040e28baaaf68a120b...,/mnt/fast/nobackup/scratch4weeks/pp01184/align...,/mnt/fast/nobackup/scratch4weeks/pp01184/align...,113287,5000,566742,25011,4d3b716d143b748b70c9125d177552c298b1a6ccf617d5...,c9a960f0e38a59645c159a50b8ec8beb0bd9add9a46641...,5000,2315b82c1336d6957cc25bc55c07975f28043151638ee5...,"{'train_dev': 0, 'train_final': 0, 'dev_final'..."


,status,models
0,COMPLETE,"[openclip_vit_b32_quickgelu_openai, mobileclip..."


In [ ]:
display(Markdown('## Locally measured paired references'))
rows = [read_json(path) for path in sorted((RESULTS / 'references').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    reference_frame = pd.DataFrame(rows)
    display(reference_frame[[c for c in ['experiment_id','i2t_R@1','t2i_R@1','mean_R@1','bidirectional_pair_latency_ms','params_total_inference'] if c in reference_frame]])
else: pending('Reference evaluation has not completed.')

In [ ]:
display(Markdown('## RTX PRO 6000 Blackwell batch profiling'))
profiles = read_csv(RESULTS / 'batch_probe/profiles.csv')
common = read_json(RESULTS / 'batch_probe/common_batch.json')
if profiles.empty: pending('Batch profiling has not completed.')
else:
    display(pd.DataFrame([common]))
    display(profiles)
    for name, group in profiles.groupby('experiment_id'):
        group.plot(x='batch_size', y='images_per_second', marker='o', title=name, figsize=(7,3)); plt.show()

In [ ]:
display(Markdown('## Six-pair full-schedule comparison'))
rows = [read_json(path) for path in sorted((RESULTS / 'pair').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    pair_frame = pd.DataFrame(rows).sort_values('mean_R@1', ascending=False)
    display(pair_frame[[c for c in ['experiment_id','seed','mean_R@1','bidirectional_pair_latency_ms','params_total_inference','params_trainable_inference'] if c in pair_frame]])
else: pending('Pair training/evaluation has not completed.')
selection = read_json(RESULTS / 'selection/pair.json')
if selection: display(Markdown('### Pre-registered pair gate')); display(pd.DataFrame([selection]))

In [ ]:
display(Markdown('## MobileCLIP2 teacher cache integrity'))
cache = read_json(RESULTS / 'teacher_cache/mobileclip2_s0_dfndr2b.metadata.json')
display(pd.DataFrame([cache]) if cache else Markdown('Teacher cache pending or gated off.'))

In [ ]:
display(Markdown('## Distillation-strength sensitivity'))
rows = [read_json(path) for path in sorted((RESULTS / 'sensitivity').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    sensitivity = pd.DataFrame(rows)
    display(sensitivity.groupby('experiment_id')['mean_R@1'].agg(['mean','std','count']).reset_index())
else: pending('Distillation sensitivity is pending or gated off.')
selection = read_json(RESULTS / 'selection/distillation.json')
if selection: display(pd.DataFrame([selection]))

In [ ]:
display(Markdown('## Optional-component smoke tests and two-seed ablations'))
smoke = read_json(RESULTS / 'component_smoke.json')
if smoke: display(pd.DataFrame(smoke.get('components', [])))
rows = [read_json(path) for path in sorted((RESULTS / 'ablation').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    ablations = pd.DataFrame(rows)
    display(ablations.groupby('experiment_id')['mean_R@1'].agg(['mean','std','count']).reset_index().sort_values('mean', ascending=False))
else: pending('Ablations are pending or gated off.')
recipe = read_json(RESULTS / 'selection/recipe.json')
if recipe: display(Markdown('### Locked recipe')); display(pd.DataFrame([recipe]))

In [ ]:
display(Markdown('## Locked full-data three-seed result'))
rows = [read_json(path) for path in sorted((RESULTS / 'final').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    final = pd.DataFrame(rows)
    display(final)
    display(final[['mean_R@1','i2t_R@1','t2i_R@1']].agg(['mean','std','min','max']))
else: pending('Sealed final evaluation has not run.')
transfer_rows = [read_json(path) for path in sorted((RESULTS / 'transfer').glob('*/metrics.json'))]
transfer_rows = [row for row in transfer_rows if row]
if transfer_rows: display(Markdown('### Optional Flickr30k transfer')); display(pd.DataFrame(transfer_rows))

In [ ]:
display(Markdown('## Oracle complementarity diagnostic'))
value = read_json(RESULTS / 'oracle.json')
display(pd.DataFrame([value]) if value else Markdown('Oracle analysis pending.'))
display(Markdown('*This development-set expert oracle is an upper bound, not a learned or deployable router result.*'))

In [ ]:
display(Markdown('## Final measured report and literature context'))
measured = read_csv(RESULTS / 'measured_results.csv')
seed_statistics = read_csv(RESULTS / 'seed_statistics.csv')
literature = read_csv(RESULTS / 'literature_context.csv')
report = read_json(RESULTS / 'report.json')
if report: display(pd.DataFrame([report]))
display(Markdown('### Locally measured'))
display(measured if not measured.empty else Markdown('Measured report pending.'))
if not seed_statistics.empty:
    display(Markdown('### Seed statistics (Student-t intervals are unstable at n=2/3)'))
    display(seed_statistics)
display(Markdown('### Literature-only (not locally measured)'))
display(literature if not literature.empty else Markdown('No literature context available.'))